# Fine-Tuning Mistral-7B-Instruct-v0.3 for Enterprise Financial Document Intelligence

This notebook fine-tunes **Mistral-7B-Instruct-v0.3** using **4-bit QLoRA** to act as a grounded financial synthesizer with:
1. **Strict Citation Enforcement**: Mandatory `[Ticker, Year, Section]` citation anchors.
2. **Refusal Protocol**: Explicit refusal whenever disclosures lack evidence, eliminating financial hallucinations.
3. **Loss Masking**: Backpropagating gradients exclusively across assistant response tokens.

> **Hardware Requirement**: Google Colab Free Tier **T4 GPU** (16 GB VRAM) or any NVIDIA GPU (A10G, A100, RTX 3090/4090).

## 1. Verify GPU Acceleration
Ensure runtime is set to **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies

In [ ]:
!pip install -q -U torch transformers peft bitsandbytes accelerate datasets trl

## 3. Prepare ChatML Instruction Dataset
Creates grounded 10-K training examples with strict citation anchors and refusal cases.

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = "You are an elite, SEC-compliant Financial Document Intelligence AI Assistant. Rely strictly on provided excerpts and include mandatory [Ticker, Year, Section] citations. If disclosures are insufficient, state that the disclosures are insufficient."

training_data = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "EXCERPTS FROM VERIFIED SEC 10-K FILINGS:\nCitation Anchor: [NVDA, 2024, ITEM 1A. RISK FACTORS]\nContent:\nWe rely on third-party foundry partners, primarily Taiwan Semiconductor Manufacturing Company (TSMC), to manufacture all of our semiconductor wafers.\n\nUSER QUERY:\nWhat is NVIDIA\x27s main foundry risk in fiscal 2024?"},
            {"role": "assistant", "content": "In fiscal year 2024, NVIDIA relies exclusively on Taiwan Semiconductor Manufacturing Company (TSMC) to manufacture all of its semiconductor wafers, creating significant foundry concentration risk [NVDA, 2024, ITEM 1A. RISK FACTORS]."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "EXCERPTS FROM VERIFIED SEC 10-K FILINGS:\nCitation Anchor: [NVDA, 2024, ITEM 7. MANAGEMENT\x27S DISCUSSION AND ANALYSIS]\nContent:\nCompute & Networking revenue increased 217% to $47.4 billion in fiscal year 2024, driven by customer demand for the NVIDIA HGX platform and Hopper architecture.\n\nUSER QUERY:\nWhat drove NVIDIA\x27s Compute & Networking revenue growth in FY2024?"},
            {"role": "assistant", "content": "Compute & Networking revenue expanded 217% to $47.4 billion in fiscal 2024, driven by surging enterprise demand for the NVIDIA HGX platform and Hopper architecture [NVDA, 2024, ITEM 7. MANAGEMENT\x27S DISCUSSION AND ANALYSIS]."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "EXCERPTS FROM VERIFIED SEC 10-K FILINGS:\nCitation Anchor: [AAPL, 2024, ITEM 7. MANAGEMENT\x27S DISCUSSION AND ANALYSIS]\nContent:\nTotal net sales were $391.0 billion in fiscal 2024. Services revenue set an all-time record of $96.2 billion, up 13%.\n\nUSER QUERY:\nWhat was Apple\x27s total commercial airplane leasing revenue in 2024?"},
            {"role": "assistant", "content": "The provided SEC 10-K disclosures are insufficient to confirm this information."}
        ]
    }
]

dataset = Dataset.from_list(training_data)
print(f"Loaded {len(dataset)} instruction pairs.")

## 4. Load Base Model in 4-bit NormalFloat (NF4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Mistral-7B base model into 4-bit VRAM...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

## 5. Configure LoRA Parameters

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 6. Train with SFTTrainer and Save Adapter Weights

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./checkpoints/mistral_7b_financial_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="messages",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args
)

print("Training QLoRA adapters...")
trainer.train()

trainer.model.save_pretrained("./checkpoints/mistral_7b_financial_lora")
tokenizer.save_pretrained("./checkpoints/mistral_7b_financial_lora")
print("Training complete! Adapters saved to ./checkpoints/mistral_7b_financial_lora")

## 7. Inference Test: Grounded Citation Verification

In [ ]:
prompt = f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
EXCERPTS FROM VERIFIED SEC 10-K FILINGS:
Citation Anchor: [NVDA, 2024, ITEM 1A. RISK FACTORS]
Content:
We rely on third-party foundry partners, primarily Taiwan Semiconductor Manufacturing Company (TSMC), to manufacture all of our semiconductor wafers.

USER QUERY:
What are NVIDIA\x27s primary risks regarding semiconductor foundry concentration in fiscal 2024?<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))